> 在线讲义：[第 62 章 · Agent 任务的资源需求与运行环境：先满的是内存，不是 CPU](https://wikieden.github.io/ai-infra-from-constraints/lessons/62-agent-sandbox.html) · 本书《从约束推导 AI Infra》

# 第 62 章 · Agent 任务的资源需求与运行环境：先满的是内存，不是 CPU

配套讲义：仓库里 `ai-infra/notes/lessons/62-agent-sandbox.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



代码修复平台（每秒 10 项任务、三轮、模型 9 s、工具 1 s、2 GiB microVM、一台 m5d.metal）的资源账、
模板加载方式、暂停与每轮重建、提前准备的期望等待，取自 AII 第 11 章 §11.1–§11.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB, GiB = 2**20, 2**30
LAM = 10            # 项/s
ROUNDS, MODEL_S, TOOL_S, ENV_GIB = 3, 9.0, 1.0, 2
CORES, MEM_GIB, SLOTS = 48, 384e9 / 2**30, 320   # 384 GB ≈ 357.6 GiB

**函数 `task_time`**：小工具。

In [ ]:
def task_time(model: float = MODEL_S, tool: float = TOOL_S, rounds: int = ROUNDS) -> float:
    return rounds * (model + tool)

**函数 `demands`**：资源需求率 = 到达率 × 每任务平均需求。

In [ ]:
def demands(model: float = MODEL_S, lam: float = LAM) -> dict[str, float]:
    """资源需求率 = 到达率 × 每任务平均需求。"""
    t = task_time(model)
    return {"cpu_cores": lam * ROUNDS * TOOL_S, "mem_gib": lam * ENV_GIB * t, "model_calls": lam * ROUNDS * model}

**函数 `max_task_rate`**：内存装得下的环境数 ÷ 每项占用时间。

In [ ]:
def max_task_rate(model: float = MODEL_S) -> float:
    """内存装得下的环境数 ÷ 每项占用时间。"""
    return (MEM_GIB // ENV_GIB) / task_time(model)

**函数 `template_local_mib`**：小工具。

In [ ]:
def template_local_mib(mode: str, template: int = 2048, read: int = 512, dirty: int = 128, admin: int = 4, hot: int = 256) -> int:
    return {"完整复制": template + admin, "按需加载": read + admin, "只读共享": dirty + admin, "共享 + 热点": dirty + admin + hot}[mode]

**函数 `fetch_s`**：小工具。

In [ ]:
def fetch_s(gib: float, bw: float = 3.125e9) -> float:
    return gib * GiB / bw

**函数 `pause_saving`**：暂停后真正空出内存的秒数。

In [ ]:
def pause_saving(env_gib: float, gap: float, save_per_gib: float = 4, resume: float = 1) -> float:
    """暂停后真正空出内存的秒数。"""
    return max(0.0, gap - env_gib * save_per_gib - resume)

**函数 `rebuild_gib_s`**：小工具。

In [ ]:
def rebuild_gib_s(prep: float = 2, tool: float = TOOL_S) -> float:
    return ROUNDS * (prep + tool) * ENV_GIB

**函数 `expected_wait`**：E[W] = P − h·min(P, L)。

In [ ]:
def expected_wait(P: float, L: float, h: float) -> float:
    """E[W] = P − h·min(P, L)。"""
    return P - h * min(P, L)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：一项任务")
print(f"  三轮 × (9 + 1) = {task_time():.0f} s；工具 CPU 3 CPU·秒；环境 2 GiB × 30 s = {ENV_GIB * task_time():.0f} GiB·秒")

In [ ]:
print("\n第二步：平台需求（每秒 10 项）")
for m in (9, 12):
    d = demands(m)
    print(f"  模型 {m} s：CPU {d["cpu_cores"]:.0f}/{CORES} 核（{d["cpu_cores"] / CORES:.1%}）、内存 {d["mem_gib"]:.0f}/{MEM_GIB:.0f} GiB（{d['mem_gib'] / MEM_GIB:.0%}）、并发调用 {d['model_calls']:.0f}/{SLOTS}（{d['model_calls'] / SLOTS:.0%}）")
print(f"  内存最多 {MEM_GIB // ENV_GIB:.0f} 个环境 → 每秒只能开始 {max_task_rate():.1f} 项；Little 定律 10 × 30 = 300 个环境")

In [ ]:
print("\n第三步：模板加载（100 个环境）")
for mode in ("完整复制", "按需加载", "只读共享", "共享 + 热点"):
    v = template_local_mib(mode)
    print(f"  {mode}：每个 {v} MiB，100 个 {100 * v / 1024:.0f} GiB")
print(f"  冷路径取回 2 GiB + 加载 0.5 GiB：{fetch_s(2.5):.2f} s；热路径 {fetch_s(0.5):.2f} s；再缺页 1 GiB 多 {fetch_s(1):.2f} s")

In [ ]:
print("\n第四步：暂停与重建")
for g in (2, 0.5):
    print(f"  {g} GiB 环境、9 s 间隔：暂停能空出 {pause_saving(g, 9):.0f} s")
print(f"  32K 个 2 GiB 沙箱 = {32768 * 2 * GiB / 1e12:.0f} TB；暂停 2 小时空出时间是保存恢复的 {7200 / 9:.0f} 倍")
r = rebuild_gib_s()
print(f"  每轮重建：{ENV_GIB * task_time():.0f} → {r:.0f} GiB·秒/项，平台 {LAM * ENV_GIB * task_time():.0f} → {LAM * r:.0f} GiB；CPU 30 → {30 + 30 * 0.1:.0f} 核")

In [ ]:
print("\n第五步：提前准备（P = 2 s，h = 0.75）")
for L in (0, 1, 2, 3):
    print(f"  提前 {L} s：平均等 {expected_wait(2, L, 0.75):.2f} s")
print(f"  提前 3 s 多出的空闲占用 {0.75 * 2 * 1:.1f} GiB·秒/次")

## 3. 代码里才看得到的

1. 内存容量用 384 GB = 357.6 GiB，整除 2 得 178 个环境；写成 358 会多出一个。
2. `pause_saving` 把保存和恢复期间都算作"仍占内存"，所以 9 s 间隔、2 GiB 时正好为 0。
3. `expected_wait` 在 $L > P$ 后不再下降，多提前的部分全变成就绪后的空闲占用。

## 4. 自己改一改

- 给 `demands` 加"每轮重建"的版本，算 CPU 多出的创建开销。
- 用 `pause_saving` 找 2 GiB 环境暂停开始有收益的最短间隔。
- 把 `expected_wait` 改成猜错时还要先取消 0.2 s，看最优提前量变不变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 包间从头订到尾，菜一道道等；
> 车流不大停得久，车位先满收费员还闲；
> 合租隔断独栋，越独立越慢越占地方，还要防住户自己拆墙；
> 饭局中间退包间，手续占满间隔就白退；准备赶上开席就够。